# 🌐 Consumir APIs REST con `requests`
*(versión explicada paso a paso)*

---

Una **API** (*Application Programming Interface*) es un servicio remoto que expone datos u operaciones a través de la web. Cualquier programa puede llamarlas para obtener información en tiempo real.

En este notebook aprenderemos a:

1. Hacer una petición **GET** con la biblioteca `requests`.
2. Interpretar el **código de respuesta HTTP**.
3. Convertir la respuesta JSON en un dict de Python.
4. Navegar por estructuras JSON anidadas.

Usaremos APIs **gratuitas y sin registro** para poder practicar sin barreras.

## Requisito previo

La biblioteca `requests` es externa (no viene de serie). Se instala así:

```bash
pip install requests
```

En Jupyter, puedes ejecutarlo desde una celda con un `!` delante:

```python
!pip install requests
```

## 1. Nuestra primera API: **REST Countries**

REST Countries es una API pública, gratis, **sin API Key**, que ofrece información de todos los países del mundo. La URL base es:

```text
https://restcountries.com/v3.1/name/{nombre}
```

Vamos a preguntar por España:

In [ ]:
import requests

url = 'https://restcountries.com/v3.1/name/spain'
respuesta = requests.get(url)

print(f'Código de respuesta: {respuesta.status_code}')
print(f'¿Es 200 (OK)?       {respuesta.status_code == 200}')

### ¿Qué son los códigos HTTP?

Cada respuesta viene con un **código numérico** que indica qué ha pasado:

| Código | Significado |
|:---:|:---|
| **200** | OK — todo bien, procesa los datos |
| **400** | Bad Request — parámetros mal formados |
| **401** | Unauthorized — falta clave o es inválida |
| **404** | Not Found — el recurso no existe |
| **429** | Too Many Requests — has hecho demasiadas llamadas |
| **500** | Internal Server Error — problema del servidor |

**Siempre comprueba el código antes de procesar la respuesta.**

## 2. Convertir la respuesta JSON en dict

Cuando la API responde con JSON (lo más habitual), el método `.json()` lo convierte directamente en dict/lista de Python:

In [ ]:
if respuesta.status_code == 200:
    datos = respuesta.json()   # ← JSON → estructura Python
    print(f'Tipo: {type(datos).__name__}')
    print(f'Longitud: {len(datos)}')
else:
    print(f'❌ Error: {respuesta.status_code}')

## 3. Navegar por datos anidados

Las respuestas de APIs suelen tener **muchos niveles de anidamiento**. Se navega paso a paso:

In [ ]:
# La API devuelve una lista con un país (el primero)
espana = datos[0]

# El nombre común está en name.common
print(f"Nombre:     {espana['name']['common']}")
print(f"Oficial:    {espana['name']['official']}")
print(f"Capital:    {espana['capital'][0]}")
print(f"Región:     {espana['region']}")
print(f"Población:  {espana['population']:,}")
print(f"Superficie: {espana['area']:,.0f} km²")

# Las lenguas oficiales están en un dict
print('\nLenguas oficiales:')
for codigo, nombre in espana['languages'].items():
    print(f'  • {nombre} ({codigo})')

## 4. Petición con parámetros

Muchas APIs aceptan **parámetros** para refinar la consulta. Se pasan con el argumento `params`, que es un dict.

Ejemplo: la API OpenMeteo devuelve pronósticos meteorológicos con latitud y longitud:

In [ ]:
url = 'https://api.open-meteo.com/v1/forecast'
parametros = {
    'latitude': 38.98,      # Ciudad Real
    'longitude': -3.93,
    'current': 'temperature_2m,relative_humidity_2m',
}

respuesta = requests.get(url, params=parametros)
print(f'URL efectiva: {respuesta.url}')
print(f'Código:       {respuesta.status_code}')

In [ ]:
if respuesta.status_code == 200:
    datos = respuesta.json()
    actual = datos['current']
    print(f"Hora local:   {actual['time']}")
    print(f"Temperatura:  {actual['temperature_2m']} °C")
    print(f"Humedad:      {actual['relative_humidity_2m']}%")

## 5. APIs con API Key

Muchas APIs (OpenWeatherMap, mapas de Google, Twitter...) requieren una **API Key** para saber quién eres y controlar los abusos. La consigues **registrándote** en el servicio (gratis en muchos casos).

**Regla importante**: la API Key es **secreta**. Nunca la publiques en un repositorio de código, ni la envíes por email sin cifrar. En proyectos serios se guarda en **variables de entorno**.

Ejemplo con OpenWeatherMap (necesitarías tu propia key):

```python
url = 'https://api.openweathermap.org/data/2.5/weather'
parametros = {
    'q': 'Ciudad Real',
    'appid': 'TU_API_KEY_AQUI',
    'units': 'metric',
}
respuesta = requests.get(url, params=parametros)
```

## 6. Manejo defensivo de errores

Un programa profesional **anticipa los problemas** de la red. Estos son los patrones más útiles:

In [ ]:
import requests

def obtener_pais(nombre):
    url = f'https://restcountries.com/v3.1/name/{nombre}'
    try:
        respuesta = requests.get(url, timeout=5)   # timeout evita esperar eterno
    except requests.exceptions.Timeout:
        print('❌ La API no responde a tiempo.')
        return None
    except requests.exceptions.ConnectionError:
        print('❌ No hay conexión a Internet.')
        return None
    
    if respuesta.status_code == 200:
        return respuesta.json()
    elif respuesta.status_code == 404:
        print(f'❌ País "{nombre}" no encontrado.')
    else:
        print(f'❌ Error {respuesta.status_code}: {respuesta.text[:100]}')
    return None

datos = obtener_pais('portugal')
if datos:
    print(f"Capital de Portugal: {datos[0]['capital'][0]}")

## 🎯 Resumen: patrones de `requests`

### Petición GET simple
```python
respuesta = requests.get(url)
```

### Con parámetros
```python
respuesta = requests.get(url, params={'q': 'Madrid', 'lang': 'es'})
```

### Con timeout (recomendado)
```python
respuesta = requests.get(url, timeout=5)
```

### Procesar la respuesta
```python
if respuesta.status_code == 200:
    datos = respuesta.json()
```

## 🌟 APIs recomendadas para practicar (sin API Key)

* **[REST Countries](https://restcountries.com/)** — información de países.
* **[PokéAPI](https://pokeapi.co/)** — todo sobre Pokémon.
* **[Open-Meteo](https://open-meteo.com/)** — meteorología.
* **[Star Wars API](https://swapi.py4e.com/)** — personajes, planetas y naves.

Con API Key (registro gratis):
* **[OpenWeatherMap](https://openweathermap.org/api)** — meteorología (más rica).
* **[NASA APIs](https://api.nasa.gov/)** — imágenes astronómicas, marte, satélites.